In [0]:
# Gold Layer: Product Returns Trend by Category & Seller

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum, when, round as spark_round,
    current_timestamp, coalesce, lit, date_trunc
)

spark = SparkSession.builder.getOrCreate()

GOLD_TABLE = "ecomstore.ecomlake.gold_product_returns_trend"

# 1. Read Silver Tables
order_items = spark.read.table("ecomstore.ecomlake.silver_order_items")
products    = spark.read.table("ecomstore.ecomlake.silver_products")
returns     = spark.read.table("ecomstore.ecomlake.silver_returns")
sellers_scd = spark.read.table("ecomstore.ecomlake.silver_sellers_scd").filter(col("is_current") == True)

# 2. Build the Unified Base Dataset
# Left join ensures we keep all sales, even if they weren't returned
enriched_sales = (
    order_items.alias("oi")
    .join(products.alias("p"), on="product_id", how="left")
    .join(sellers_scd.alias("s"), on="seller_id", how="left")
    .join(returns.alias("r"), on="order_item_id", how="left")
    
    # Extract the month to establish the 'Trend' timeline
    .withColumn("sales_month", date_trunc("month", col("oi.created_at")))
)

# 3. Aggregate Metrics by Month, Category, and Seller
returns_trend = (
    enriched_sales
    .groupBy("sales_month", "p.category", "oi.seller_id", "s.seller_name")
    .agg(
        # Volume Metrics
        count("oi.order_item_id").alias("total_units_sold"),
        count("r.return_id").alias("total_units_returned"),
        
        # Reason Breakdown (Useful for root-cause analysis)
        count(when(col("r.return_reason") == "damaged", True)).alias("damaged_returns"),
        count(when(col("r.return_reason") == "wrong_item", True)).alias("wrong_item_returns"),
        count(when(col("r.return_reason") == "not_as_described", True)).alias("quality_returns"),
        
        # Financial Impact
        spark_sum("oi.total_price").alias("gross_sales_revenue"),
        spark_sum(coalesce(col("r.refund_amount"), lit(0.0))).alias("total_refunded_amount")
    )
    
    # 4. Safe Math & KPI Generation
    .withColumn("return_rate_pct",
        when(col("total_units_sold") > 0, 
             spark_round((col("total_units_returned") / col("total_units_sold")) * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("revenue_lost_pct",
        when(col("gross_sales_revenue") > 0, 
             spark_round((col("total_refunded_amount") / col("gross_sales_revenue")) * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("gross_sales_revenue", spark_round(col("gross_sales_revenue"), 2))
    .withColumn("total_refunded_amount", spark_round(col("total_refunded_amount"), 2))
    .withColumn("_gold_processed_at", current_timestamp())
    
    # Order chronologically for clean visualization
    .orderBy("sales_month", "category", col("return_rate_pct").desc())
)

# 5. Write to Gold
(
    returns_trend
    .coalesce(1) 
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact", "true")
    .saveAsTable(GOLD_TABLE)
)